# Gurbani Guidance — Vector Embedding Builder

Run this notebook on an **A100 GPU** to build the ChromaDB vector index from the SGGS PDF.

**Output:** `data/chroma/` — download and place it in your repo before running the API server.

---

### Steps
1. Install deps
2. Clone repo / upload PDF
3. Parse PDF → `data/shabads.jsonl`
4. Embed with `BAAI/bge-m3` on GPU → `data/chroma/`
5. Zip and download

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'No GPU detected')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
%pip install -q pymupdf chromadb sentence-transformers rank-bm25 pydantic python-dotenv

In [ ]:
# ── Cell 3: Clone repo (skip if already present) ─────────────────────────────
import os

REPO_URL = "https://github.com/akashdatageek/Gurbani-Guidance.git"
REPO_DIR = "Gurbani-Guidance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"{REPO_DIR} already exists — pulling latest")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# ── Cell 4: Confirm PDF is present ───────────────────────────────────────────
PDF_PATH = "src/SriGuruGranthSahibJiDarpanEnglish.pdf"

if os.path.exists(PDF_PATH):
    size_mb = os.path.getsize(PDF_PATH) / 1e6
    print(f"PDF found: {PDF_PATH} ({size_mb:.1f} MB)")
else:
    # If not in repo, upload it manually or via Google Drive
    raise FileNotFoundError(
        f"{PDF_PATH} not found.\n"
        "Upload it to the notebook environment or mount Google Drive."
    )

In [ ]:
# ── Cell 5: Parse PDF → data/shabads.jsonl (~9 seconds) ──────────────────────
import time

os.makedirs("data", exist_ok=True)

if os.path.exists("data/shabads.jsonl"):
    print("data/shabads.jsonl already exists — skipping ingest")
else:
    t0 = time.time()
    !python -m src.ingest_pdf
    print(f"Ingest done in {time.time() - t0:.1f}s")

# Quick sanity check
with open("data/shabads.jsonl") as f:
    count = sum(1 for _ in f)
print(f"Shabads: {count:,}")

In [ ]:
# ── Cell 6: GPU-accelerated embedding ────────────────────────────────────────
import json
import logging
import time

import chromadb
import torch
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
logger = logging.getLogger()

# ── Config ────────────────────────────────────────────────────────────────────
EMBED_MODEL   = "BAAI/bge-m3"
CHROMA_DIR    = "data/chroma"
COLLECTION    = "sggs"
SHABADS_FILE  = "data/shabads.jsonl"
WINDOW_SIZE   = 12
WINDOW_OVERLAP = 2
BATCH_SIZE    = 256   # A100 80 GB can handle 256+ comfortably

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 7: Load shabads and build windows ────────────────────────────────────
from src.corpus import load_shabads, make_windows

def passage_text(window):
    parts = []
    for line in window:
        segments = []
        if line.transliteration:
            segments.append(line.transliteration)
        if line.translation_en:
            segments.append(line.translation_en)
        parts.append(" | ".join(segments) if segments else line.gurmukhi)
    return "\n".join(parts)

def passage_doc(window):
    return json.dumps({
        "gurmukhi": [l.gurmukhi for l in window],
        "translation_en": [l.translation_en for l in window],
        "line_angs": [l.ang for l in window],
    }, ensure_ascii=False)

ids, texts, documents, metadatas = [], [], [], []

for shabad in load_shabads(SHABADS_FILE):
    for win_idx, window in enumerate(make_windows(shabad.lines,
                                                   window_size=WINDOW_SIZE,
                                                   overlap=WINDOW_OVERLAP)):
        ids.append(f"{shabad.shabad_id}-{win_idx}")
        texts.append(passage_text(window))
        documents.append(passage_doc(window))
        metadatas.append({
            "shabad_id": shabad.shabad_id,
            "ang": shabad.ang,
            "raag": shabad.raag,
            "writer": shabad.writer,
            "window": win_idx,
        })

print(f"Total passages to embed: {len(ids):,}")

In [ ]:
# ── Cell 8: Load model on GPU ─────────────────────────────────────────────────
print(f"Loading {EMBED_MODEL} on {DEVICE} …")
t0 = time.time()
model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print(f"Model loaded in {time.time() - t0:.1f}s")

In [ ]:
# ── Cell 9: Create ChromaDB collection ───────────────────────────────────────
os.makedirs(CHROMA_DIR, exist_ok=True)
client_chroma = chromadb.PersistentClient(path=CHROMA_DIR)

# Drop and recreate for a clean build
try:
    client_chroma.delete_collection(COLLECTION)
    print(f"Dropped existing '{COLLECTION}' collection")
except Exception:
    pass

collection = client_chroma.get_or_create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection '{COLLECTION}' ready")

In [ ]:
# ── Cell 10: Embed + upsert in batches ───────────────────────────────────────
from tqdm.auto import tqdm

total = len(ids)
t0 = time.time()

for start in tqdm(range(0, total, BATCH_SIZE), desc="Embedding"):
    end = min(start + BATCH_SIZE, total)
    batch_texts = texts[start:end]

    embeddings = model.encode(
        batch_texts,
        normalize_embeddings=True,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
    ).tolist()

    collection.upsert(
        ids=ids[start:end],
        embeddings=embeddings,
        documents=documents[start:end],
        metadatas=metadatas[start:end],
    )

elapsed = time.time() - t0
print(f"\nDone! {total:,} passages in {elapsed:.1f}s ({total/elapsed:.0f} passages/sec)")
print(f"Collection size: {collection.count():,}")

In [ ]:
# ── Cell 11: Smoke test ───────────────────────────────────────────────────────
query = "What does Gurbani say about haumai?"
q_vec = model.encode([query], normalize_embeddings=True).tolist()
results = collection.query(query_embeddings=q_vec, n_results=3)

print(f"Query: {query}\n")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), 1):
    data = json.loads(doc)
    print(f"[{i}] Ang {meta['ang']} · {meta['raag']} · {meta['writer']}")
    print(f"     {data['gurmukhi'][0][:80]}")
    print()

In [ ]:
# ── Cell 12: Zip and download ─────────────────────────────────────────────────
import shutil

print("Zipping data/ folder …")
shutil.make_archive("gurbani_data", "zip", ".", "data")
size_mb = os.path.getsize("gurbani_data.zip") / 1e6
print(f"gurbani_data.zip ready ({size_mb:.1f} MB)")

# If running on Colab:
try:
    from google.colab import files
    files.download("gurbani_data.zip")
except ImportError:
    print("Not on Colab — download gurbani_data.zip manually from the file browser.")